# Unit 3 – Multiple linear regression
**Aim:** predict house price from several features, interpret coefficients, and test what happens when predictors are dropped or become collinear.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(42)
plt.rcParams['figure.figsize'] = (6, 4)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
n = 300
rng = np.random.RandomState(42)
size = rng.normal(120, 30, n)                    # m2
bedrooms = np.clip(np.round(size / 40 + rng.normal(0, .6, n)), 1, 6)
age = rng.uniform(0, 50, n)                      # years
price = 50 + 2.5 * size + 8 * bedrooms - 1.2 * age + rng.normal(0, 20, n)   # thousands
df = pd.DataFrame({'size_m2': size, 'bedrooms': bedrooms, 'age_years': age, 'price_k': price})
print(df.corr().round(2))

def fit_report(cols):
    Xtr, Xte, ytr, yte = train_test_split(df[cols], df['price_k'], test_size=.25, random_state=42)
    m = LinearRegression().fit(Xtr, ytr)
    r2 = r2_score(yte, m.predict(Xte))
    adj = 1 - (1 - r2) * (len(yte) - 1) / (len(yte) - len(cols) - 1)
    return m, round(r2, 3), round(adj, 3)

m, r2, adj = fit_report(['size_m2', 'bedrooms', 'age_years'])
print('\nIntercept:', round(m.intercept_, 2))
print({k: round(float(v), 3) for k, v in zip(['size_m2', 'bedrooms', 'age_years'], m.coef_)})
print('Test R2 =', r2, ' adjusted R2 =', adj)

           size_m2  bedrooms  age_years  price_k
size_m2       1.00      0.76       0.01     0.95
bedrooms      0.76      1.00      -0.04     0.77
age_years     0.01     -0.04       1.00    -0.21
price_k       0.95      0.77      -0.21     1.00

Intercept: 54.78
{'size_m2': 2.484, 'bedrooms': 9.372, 'age_years': -1.345}
Test R2 = 0.926  adjusted R2 = 0.923


## Experiment 1 – drop predictors

In [3]:
rows = []
for cols in (['size_m2'], ['size_m2', 'bedrooms'], ['size_m2', 'age_years'], ['size_m2', 'bedrooms', 'age_years']):
    _, r2, adj = fit_report(cols); rows.append({'predictors': ', '.join(cols), 'test_R2': r2, 'adj_R2': adj})
pd.DataFrame(rows)

,predictors,test_R2,adj_R2
0,size_m2,0.894,0.892
1,"size_m2, bedrooms",0.900,0.897
2,"size_m2, age_years",0.925,0.923
3,"size_m2, bedrooms, age_years",0.926,0.923


## Experiment 2 – add a useless random feature and a near-duplicate feature (multicollinearity)

In [4]:
df['noise_feature'] = rng.normal(size=n)
df['size_ft2'] = df['size_m2'] * 10.764 + rng.normal(0, 1, n)   # almost the same information as size_m2
for cols in (['size_m2', 'bedrooms', 'age_years', 'noise_feature'],):
    _, r2, adj = fit_report(cols); print('with noise feature  ->', 'R2', r2, 'adj R2', adj)

# coefficient stability when size is duplicated
coefs = []
for seed in range(5):
    sample = df.sample(frac=.7, random_state=seed)
    mm = LinearRegression().fit(sample[['size_m2', 'size_ft2', 'bedrooms', 'age_years']], sample['price_k'])
    coefs.append(mm.coef_.round(3))
print('\nCoefficients [size_m2, size_ft2, bedrooms, age] across 5 resamples:')
print(pd.DataFrame(coefs, columns=['size_m2', 'size_ft2', 'bedrooms', 'age']))

with noise feature  -> R2 0.926 adj R2 0.922

Coefficients [size_m2, size_ft2, bedrooms, age] across 5 resamples:
   size_m2  size_ft2  bedrooms    age
0  -17.522     1.860     7.929 -1.239
1  -31.369     3.152     6.896 -1.302
2  -15.254     1.653     8.275 -1.254
3  -10.692     1.225     9.100 -1.178
4  -18.014     1.902     9.349 -1.296


## Observations
* Size correlates 0.95 with price, bedrooms 0.77 and age -0.21. Bedrooms and size are themselves correlated (0.76), a warning of multicollinearity.
* Size alone gives test R² = 0.894; adding age lifts it to 0.925; adding bedrooms on top adds almost nothing (0.926), because it mostly repeats information in size.
* Adding a pure-noise feature did not raise adjusted R² (0.922 vs 0.923) – adjusted R² penalises useless predictors.
* When size is entered twice (m² and ft²), the individual coefficients become unstable across resamples (size_m2 ranged from -31.4 to -10.7 while the true effect is +2.5). Predictions stay accurate, but **the coefficients can no longer be interpreted**.